In [ ]:
# Colab/bootstrap: clone this repository and install it editable.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
QWEN35_TRANSFORMERS_REVISION = "b70d02fc724d04c916832ca4ead03ff05e8fb1ee"
qwen35_probe = subprocess.run(
    [sys.executable, "-c", "from transformers import AutoModelForMultimodalLM"],
    capture_output=True,
)
if qwen35_probe.returncode != 0:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"transformers @ git+https://github.com/huggingface/transformers.git@{QWEN35_TRANSFORMERS_REVISION}",
        "torchvision", "pillow",
    ])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("ready:", root)


# Qwen-Scope Activation MVP

이 노트북은 Qwen3.5 계열에서 activation을 뽑고 Qwen-Scope SAE로 해석 후보 layer 하나를 고르는 최소 흐름입니다.

## 세부 고려사항

1. 모델과 SAE는 checkpoint, hidden size, layer index가 맞아야 합니다. `ANALYSIS_PROFILE_KEY`를 바꾸면 Qwen3.5의 대응 분석 checkpoint와 공식 SAE가 함께 선택됩니다.
2. Qwen-Scope SAE는 residual stream hook point용이므로 `model.language_model.layers.{layer}.output`을 캡처합니다.
3. NNsight trace 안에서 나중에 볼 tensor는 반드시 `.save()` 해야 합니다.
4. 첫 MVP는 `last` token activation만 봅니다. 토큰별 feature를 보고 싶으면 `TOKEN_POSITION = "all"`로 바꿔 다시 실행합니다.

In [ ]:
from pathlib import Path
import os
import sys

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
import torch
from IPython.display import display

from mindscopex_analysis import (
    DEFAULT_ANALYSIS_PROFILE_KEY,
    capture_layer_residuals,
    count_layers,
    default_sae_device,
    dtype_from_name,
    get_qwen35_analysis_profile,
    load_qwen_language_model,
    load_qwen_scope_sae,
    prepend_final_answer_instruction,
    recommended_dtype_name,
    scan_qwen_scope_layers,
    summarize_qwen_scope_features,
    top_qwen_scope_features,
)


In [ ]:
ANALYSIS_PROFILE_KEY = DEFAULT_ANALYSIS_PROFILE_KEY  # 2b, 9b, 27b, 35b-a3b
PROFILE = get_qwen35_analysis_profile(ANALYSIS_PROFILE_KEY)
MODEL_ID = PROFILE.analysis_model_id
SAE_REPO_ID = PROFILE.sae_repo_id
LAYERS = list(PROFILE.scan_layers)
TOKEN_POSITION = "last"
DTYPE = recommended_dtype_name()
SAE_DEVICE = default_sae_device()
SAE_DTYPE = DTYPE

PROMPTS = [
    (
        "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
        "How much does the ball cost? Answer in cents."
    ),
    (
        "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take "
        "100 machines to make 100 widgets? Answer in minutes."
    ),
    (
        "In a lake, a patch of lily pads doubles in size every day. If it takes 48 days "
        "to cover the whole lake, how long to cover half the lake?"
    ),
    (
        "How many animals of each kind did Moses take on the ark? "
        "Answer with a number or a short correction."
    ),
    (
        "All cats are animals. Some animals can fly. Therefore, can we conclude that "
        "some cats can fly? Answer yes or no."
    ),
]
PROMPTS = [prepend_final_answer_instruction(prompt) for prompt in PROMPTS]

print({
    "profile": PROFILE.key,
    "model": MODEL_ID,
    "sae_repo": SAE_REPO_ID,
    "layers": LAYERS,
    "dtype": DTYPE,
    "sae_device": SAE_DEVICE,
    "sae_matches_behavior_model": PROFILE.sae_matches_behavior_model,
})


In [ ]:
lm = load_qwen_language_model(
    MODEL_ID,
    device_map="auto",
    dtype=DTYPE,
    dispatch=True,
)

print("n_layers:", count_layers(lm))


In [ ]:
TARGET_LAYER = LAYERS[len(LAYERS) // 2]

residuals = capture_layer_residuals(
    lm,
    PROMPTS[:2],
    TARGET_LAYER,
    token_position=TOKEN_POSITION,
)

print("layer:", TARGET_LAYER)
print("activation shape:", tuple(residuals.shape))


In [ ]:
sae = load_qwen_scope_sae(
    SAE_REPO_ID,
    TARGET_LAYER,
    device=SAE_DEVICE,
    dtype=dtype_from_name(SAE_DTYPE),
)

summary = summarize_qwen_scope_features(residuals, sae, batch_size=64)
top_features = top_qwen_scope_features(summary, top_n=15, metric="mean_abs")
display([
    {
        "rank": i,
        "feature_id": item.feature_id,
        "mean_abs": item.mean_abs,
        "max": item.max,
        "activation_rate": item.activation_rate,
    }
    for i, item in enumerate(top_features, start=1)
])

del sae
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
scan = scan_qwen_scope_layers(
    lm,
    PROMPTS,
    LAYERS,
    repo_id=SAE_REPO_ID,
    token_position=TOKEN_POSITION,
    sae_device=SAE_DEVICE,
    sae_dtype=SAE_DTYPE,
    batch_size=64,
    top_n=15,
    metric="mean_abs",
)

display(scan.layer_rows())
best = scan.best
print("selected layer:", best.layer)
display(best.feature_rows())


## 다음 체크포인트

- `TOKEN_POSITION = "all"`로 바꿔 feature가 어느 토큰에서 켜지는지 확인합니다.
- lure/control prompt 쌍을 늘려 layer별 feature 안정성을 다시 봅니다.
- 선택된 feature의 decoder direction으로 steering 또는 activation patching을 붙여 실제 인과 효과를 검증합니다.